# Intro to Arize Phoenix 

## Tracing 

### Start Ollama Server for Inference

As we have done more other sessions, we will be using Ollama as our inference server.  Let's start the Ollama server. 

In [ ]:
# Launch Ollama serve in background, pipe output to log file
import subprocess
import os
import signal
import time
import atexit

# Create log file
log_file = "ollama_server.log"

# Kill any existing ollama processes
subprocess.run(["pkill", "ollama"], capture_output=True)
time.sleep(2)

# Start ollama serve in background
print("🚀 Starting Ollama server...")
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open(log_file, "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setpgrp
)
print(f"📄 Server logs: {log_file}")
print(f"📍 API endpoint: http://localhost:11434")

# Wait for server to start
print("⏳ Waiting 5 seconds for server startup...")
time.sleep(5)

# Store process PID globally for cleanup (avoid %store magic)
if 'OLLAMA_PROCESS' not in globals():
    OLLAMA_PROCESS = process.pid
    print(f"✅ Ollama server ready! PID: {OLLAMA_PROCESS}")
else:
    print("✅ Ollama server already running!")

# Register cleanup function
def cleanup_ollama():
    try:
        if 'OLLAMA_PROCESS' in globals():
            os.killpg(os.getpgid(OLLAMA_PROCESS), signal.SIGTERM)
            print("🛑 Ollama server stopped")
    except:
        pass

atexit.register(cleanup_ollama)

### Start Phoenix Server

This cell sets up and launches a Phoenix session within the Jupyter notebook environment.

- It first imports the required packages — phoenix for model monitoring, os for working with environment variables and directories, and nest_asyncio to allow asynchronous event loops to run properly inside Jupyter.
- nest_asyncio.apply() ensures that Phoenix’s async server can run smoothly alongside the notebook’s internal event loop.
- Next, it creates a local folder called phoenix_data to store session data persistently and then sets environment variables (PHOENIX_WORKING_DIR and PHOENIX_PORT) to point Phoenix to this directory and to define which port the web UI will use.
- Finally, it launches the Phoenix web app with px.launch_app(). Setting use_temp_dir=True means it will use a temporary directory for session data (use False if you want to keep records across sessions).
- The printed message reminds you to open the Phoenix dashboard using the URL shown when running the phoenix_setup.ipynb notebook

In [ ]:
import phoenix as px
import os
import nest_asyncio
nest_asyncio.apply()

# start phoenix session for the notebook

# create directory and environtment variables for persistent storage 
os.makedirs('phoenix_data', exist_ok=True)
os.environ['PHOENIX_WORKING_DIR'] = 'phoenix_data'

# set port to that specified in port forwarding and launch server 
os.environ["PHOENIX_PORT"] = "5903"
session = px.launch_app(use_temp_dir=False)  # switch to False for consistent storage
print('PLEASE USE URL SPECIFIED IN phoenix_setup.ipynb TO VIEW PHOENIX SERVER')

### Setup Tracing 

This cell configures OpenTelemetry tracing so Phoenix can monitor and visualize activity from your OpenAI API calls.

- It imports register from phoenix.otel, which helps connect Phoenix to your tracing system, and OpenAIInstrumentor from the OpenInference library, which automatically instruments OpenAI API calls.
- The register() function creates and registers a tracer provider for your project — in this case named "first-ollama-tracing-demo". It also points this tracer to your local Phoenix endpoint so the traces will appear in your Phoenix dashboard.
- OpenAIInstrumentor().instrument(tracer_provider=tracer_provider) automatically attaches tracing hooks to all OpenAI API operations, so requests and responses are captured without any extra code changes.
- Finally, tracer = tracer_provider.get_tracer(__name__) retrieves a named tracer instance that you can use later in your code if you want to create custom spans or trace additional logic manually.

In [ ]:
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
#from opentelemetry import trace

# Register tracer (use local Phoenix endpoint)
tracer_provider = register(
    project_name="first-ollama-tracing-demo"
)
# automatically traces any API calls to open AI with no additional work
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = tracer_provider.get_tracer(__name__)

### A first tracing demo

Ok, Let's see what happens  when we use OpenAI to chat with models.  Run the following cell then, view the phoenix server to see if the interaction was traced.

In [ ]:
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",  # Ollama endpoint
    api_key="ollama"  # Dummy key, unused by Ollama
)

response = ollama_client.chat.completions.create(
    model="llama3.2:latest",
    messages=[{"role": "user", "content": "Why is the sky blue?"}]
)
print(response.choices[0].message.content)

### Tracing agentic systems 

Next let's try running the agentic framework you ran in the last session.  This code base already is set up to trace its outputs with Arize Phoenix.  First let's import a few things to get the agent framework running

In [ ]:
import sys
import os

# Resolve "../TACC_exAI" relative to the current working directory
new_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if new_path not in sys.path:
    sys.path.insert(0, new_path)  # or .append(new_path)

### A few notes on tracing

This code block creates a tracing span around the entire Agent execution so Phoenix can track and visualize the complete workflow.

- with tracer.start_as_current_span("AgentRun", openinference_span_kind="agent") as span: creates a new span named "AgentRun" marked as an agent type operation. This span acts as a container that tracks timing, inputs, and outputs for the entire agent run, making it visible in your Phoenix dashboard.
- span.set_input(prompt) records the input prompt as structured data within this span, so you can see exactly what the agent was asked to process.
- span.set_attribute(result,"result from agent.run") attaches the agent's final output as an attribute to the same span, linking the input directly to the result for easy debugging and performance analysis.
- The with block ensures the span automatically closes when the agent finishes, capturing the total execution time.

In [ ]:
#from TACC_exAI.agent import Agent

#prompt = "Hi! What are the first 20 digits of the fibonacci sequence?"

#with tracer.start_as_current_span("AgentRun", openinference_span_kind="agent") as span:
#    span.set_input(prompt)
#    agent = Agent(
#            mode="dev",
#            session=None,
#            experiment=True,
#            experiment_prompt="Hi! What are the first 20 digits of the fibonacci sequence?",
#           default_action_model_name="qwen3:32b",
#           use_apptainer=True, 
#           tracer=tracer
#       )
    
#    result = agent.run()
#    span.set_attribute(result,"result from agent.run")

## Arize Phoenix: Experiments  

Arize Phoenix Experiments are a set of tools within the Arize Phoenix platform that enable structured evaluation and optimization of AI agents by running targeted experiments on organized datasets, tracking critical metrics, and facilitating iterative improvement cycles.  In short, the are a great tool for keeping track of various benchmarks of your LLM application 

### Experiments Introduction 

You can set up experiments to evaluate your agentic system. See [overview page](https://arize.com/docs/ax/develop/datasets-and-experiments).

These experiments consist of the following:

1. A **dataset** which is a pandas.DataFrame with questions, expected answers and additional metadata.
2. A **task** which is a minimal function that executes your LLM application and returns an output.
3. The **evaluator** which does some sort of evaluation like compares the actual task output to the expected output
4. A function call to **run_experiment** executes the evaluation workflow and records results.

 
![](experiment1.png)

[Image source](https://learn.deeplearning.ai/courses/evaluating-ai-agents/lesson/x3i1d/adding-router-and-skill-evaluations)

Below is an overly simple example of what this workflow might look like for the simplest version of an LLM chatbot. 

In [ ]:
from phoenix.client import Client
import pandas as pd
from phoenix.experiments import run_experiment

px_client = px.Client()

### Upload dataset 

This code creates a small question–answer dataset using pandas and uploads it to a platform via a px_client instance.

First, a DataFrame is defined with two columns: "question" (the input text) and "expected" (the correct answers). It includes a mix of simple math, general knowledge, logic, and word tasks.

Then, the dataset is uploaded using px_client.upload_dataset(). The parameters tell the system that "question" is the input field and "expected" is the output field, giving the dataset the name "simple-qa-benchmark".

In short, the script builds and registers a mini benchmark Q&A dataset for testing or evaluation.

After running this cell, lets view the phoenix server and see our newly uploaded dataset.

In [ ]:
# Simple Q&A dataset
df = pd.DataFrame({
    "question": [
        "What is 2+2?",
        "Capital of France?", 
        "Largest planet?",
        "If today is Monday, what day was yesterday?",
        "If all cats are mammals and some mammals fly, do some cats fly?",
        "Repeat the word 'no' three times, but as one word.",
    ],
    "expected": [
        "4",
        "Paris",
        "Jupiter",
        "Sunday",
        "No",
        "nonono",
    ]
})

# Upload dataset (input_key="question")
dataset = px_client.upload_dataset(
    dataframe=df,
    input_keys=["question"],
    output_keys=["expected"],
    dataset_name="simple-qa-benchmark",
)

### Define task and evaluator functions 

This code defines two functions — one to generate answers using a language model and another to check if those answers are correct.

The task() function takes an example from the dataset and sends its "question" value to the ollama_client, asking a model named "llama3.2:latest" to reply with exactly one English word and no punctuation. It retrieves the model’s reply, cleans it up, and returns it as {"answer": answer}.

The exact_match() function then compares the model’s answer to the expected answer from the dataset.



In [ ]:
def task(example):
    question = example.input["question"]

    response = ollama_client.chat.completions.create(
        model="llama3.2:latest",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using exactly one English word and nothing else. "
                    "Do not include punctuation or explanations."
                ),
            },
            {"role": "user", "content": question}
        ],
    )

    answer = response.choices[0].message.content.strip().split()[0]
    return {"answer": answer}
    
def exact_match(output, expected):
    """Returns True if answer matches expected"""
    return output["answer"] == expected["expected"]

### Launch experiment with Arize built in method

In [ ]:
# One line runs the full experiment!
experiment = run_experiment(
    dataset=dataset,
    task=task,
    evaluators=[exact_match],
    experiment_name="simple-qa-benchmark-run",
    concurrency=1
)

## Benchmarks across multiple llm-app implementations 

One really nice feature of the experiments is this allows us to run systematic benchmarks across our software where we may be changing prompt templates, models used, or versions of software we are running 

![](experiment2.png)

Let's try changing the model we are using in our benchmark and see how results change. 

In [ ]:
def alt_task(example):
    question = example.input["question"]

    response = ollama_client.chat.completions.create(
        model="llama3.1:8b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using exactly one English word and nothing else. "
                    "Do not include punctuation or explanations."
                ),
            },
            {"role": "user", "content": question}
        ],
    )

    answer = response.choices[0].message.content.strip().split()[0]
    return {"answer": answer}

experiment = run_experiment(
    dataset=dataset,
    task=alt_task,
    evaluators=[exact_match],
    experiment_name="simple-qa-benchmark-run with llama3.1",
    concurrency=1
)

## Exercise: 

Try changing the system prompt and re-run the experiment to see how your results change.  Your change in prompt template can be inspired by the results of previous runs.